# Data Exploration

Quick look at the summarization dataset: load a sample, tokenize it,
inspect sequence-length distributions, and sanity-check the vocabulary
encode/decode round trip before kicking off training.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from src.preprocessing.tokenizer import SimpleTokenizer
from src.preprocessing.vocabulary import Vocabulary
from src.preprocessing.dataset import SummarizationDataset

## Load a sample of the dataset

Point `DATA_PATH` at a CSV/JSON file with `article` and `highlights`
columns (the CNN/DailyMail convention) once you've added one to `data/`.

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "train.csv"

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
    print(df.shape)
    display(df.head())
else:
    print(f"No dataset found at {DATA_PATH} yet -- add one to follow along.")

## Tokenize a few examples

In [ ]:
tokenizer = SimpleTokenizer()

if DATA_PATH.exists():
    sample_article = df.iloc[0]["article"]
    sample_summary = df.iloc[0]["highlights"]
    print("Article tokens:", tokenizer.tokenize(sample_article)[:30])
    print("Summary tokens:", tokenizer.tokenize(sample_summary)[:30])

## Sequence length distribution

Useful for choosing `max_src_len` / `max_tgt_len` for training.

In [ ]:
if DATA_PATH.exists():
    article_lens = df["article"].apply(lambda t: len(tokenizer.tokenize(t)))
    summary_lens = df["highlights"].apply(lambda t: len(tokenizer.tokenize(t)))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(article_lens, bins=50)
    axes[0].set_title("Article length (tokens)")
    axes[1].hist(summary_lens, bins=50)
    axes[1].set_title("Summary length (tokens)")
    plt.tight_layout()
    plt.show()

    print(article_lens.describe())
    print(summary_lens.describe())

## Build a vocabulary and check the encode/decode round trip

In [ ]:
if DATA_PATH.exists():
    texts = SummarizationDataset.texts(DATA_PATH)
    tokenized = (tokenizer.tokenize(t) for t in texts)
    vocab = Vocabulary.build(tokenized, min_freq=2, max_size=30000)
    print(f"Vocabulary size: {len(vocab)}")

    ids = vocab.encode(tokenizer.tokenize(sample_summary))
    decoded = vocab.decode(ids)
    print("Round-trip:", tokenizer.detokenize(decoded))